# Imports

In [ ]:
import os
import cv2
import torch
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# Mount Google Drive (Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Device

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


# Dataset Class

In [ ]:
class KeypointsDataset(Dataset):

    def __init__(self, image_dir, label_dir):

        self.image_dir = image_dir
        self.label_dir = label_dir

        self.image_files = sorted(
            [f for f in os.listdir(image_dir)
             if f.endswith(('.jpg','.jpeg','.png'))]
        )

        self.transforms = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225]
            )
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        image_name = self.image_files[idx]

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transforms(img)

        label_name = os.path.splitext(image_name)[0] + ".txt"

        label_path = os.path.join(
            self.label_dir,
            label_name
        )

        with open(label_path, "r") as f:
            values = list(map(float, f.readline().split()))

        kp_values = values[5:]

        kps = []

        for i in range(0, len(kp_values), 3):

            x = kp_values[i]
            y = kp_values[i + 1]

            x *= 224
            y *= 224

            kps.extend([x, y])

        kps = np.array(kps, dtype=np.float32)

        return img, kps

# Dataset Paths

In [ ]:
!pip install roboflow
!pip install ultralytics

from roboflow import Roboflow
rf = Roboflow(api_key="8npWhUYJ1aeLSwla4BLf")
project = rf.workspace("squash-project").project("squash-court-keypoint-detection")
version = project.version(3)
dataset = version.download("yolov8")



loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Squash-Court-Keypoint-Detection-3 in yolov8:: 100%|██████████| 196/196 [00:00<00:00, 6310.52it/s]

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
train_dataset = KeypointsDataset(
    "Squash-Court-Keypoint-Detection-3/train/images",
    "Squash-Court-Keypoint-Detection-3/train/labels"
)

val_dataset = KeypointsDataset(
    "Squash-Court-Keypoint-Detection-3/valid/images",
    "Squash-Court-Keypoint-Detection-3/valid/labels"
)

# Dataloaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

# Check Dataset

In [ ]:
img, kps = train_dataset[0]

print(img.shape)
print(kps.shape)
print(kps)

torch.Size([3, 224, 224])
(18,)
[     67.053      111.02      66.868      101.19      66.132      67.134      64.289     0.32748      57.658      164.07         112      163.09      54.342      184.05      166.53      163.74      169.66      182.74]


# Model

In [ ]:
model = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V2
)

model.fc = torch.nn.Linear(
    model.fc.in_features,
    18
)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 192MB/s]


# Loss and Optimizer

In [ ]:
criterion = torch.nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

# Checkpoint Directory

In [ ]:
checkpoint_dir = "/content/drive/MyDrive/squash_checkpoints"

os.makedirs(
    checkpoint_dir,
    exist_ok=True
)

best_val_loss = float('inf')

# Training

In [ ]:
epochs = 100

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for imgs, kps in train_loader:

        imgs = imgs.to(device)
        kps = kps.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)

        loss = criterion(outputs, kps)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for imgs, kps in val_loader:

            imgs = imgs.to(device)
            kps = kps.to(device)

            outputs = model(imgs)

            loss = criterion(outputs, kps)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"epoch_{epoch+1}.pth"
    )

    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss
    }, checkpoint_path)

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss
        },
        os.path.join(
            checkpoint_dir,
            "best_model.pth"
        ))

        print("Best model saved!")

Epoch 1/100 | Train Loss: 9517.1087 | Val Loss: 7021.6867
Best model saved!
Epoch 2/100 | Train Loss: 7642.1562 | Val Loss: 5725.1396
Best model saved!
Epoch 3/100 | Train Loss: 6300.4140 | Val Loss: 5561.2555
Best model saved!
Epoch 4/100 | Train Loss: 5282.3801 | Val Loss: 5510.6874
Best model saved!
Epoch 5/100 | Train Loss: 4364.0811 | Val Loss: 5005.2063
Best model saved!
Epoch 6/100 | Train Loss: 3752.6739 | Val Loss: 4242.6801
Best model saved!
Epoch 7/100 | Train Loss: 3071.1077 | Val Loss: 3614.8891
Best model saved!
Epoch 8/100 | Train Loss: 2661.0364 | Val Loss: 4224.6281
Epoch 9/100 | Train Loss: 2282.5785 | Val Loss: 2695.3665
Best model saved!
Epoch 10/100 | Train Loss: 1942.7217 | Val Loss: 2806.0467
Epoch 11/100 | Train Loss: 1624.4917 | Val Loss: 2642.0476
Best model saved!
Epoch 12/100 | Train Loss: 1378.2553 | Val Loss: 1685.8612
Best model saved!
Epoch 13/100 | Train Loss: 1181.7269 | Val Loss: 2056.0674
Epoch 14/100 | Train Loss: 1013.7889 | Val Loss: 1732.2460
Epo

# Final Save

In [ ]:
torch.save(
    model.state_dict(),
    "squash_court_keypoints.pth"
)

print("Training Complete")

Training Complete
